## 1) Objective

This notebook extends the raw-data overview with deeper data understanding:
- table-level summary and schema checks
- missing-value analysis
- basic distribution plots for order behavior
- quick quality checks before preparation/EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Resolve data directory robustly whether notebook runs from project root or notebook folder
candidate_dirs = [Path('data/raw'), Path('../../data/raw')]
data_dir = next((p for p in candidate_dirs if p.exists()), None)
if data_dir is None:
    raise FileNotFoundError('Cannot find data/raw directory from current working directory.')

order_products_prior = pd.read_csv(data_dir / 'order_products__prior.csv')
order_products_train = pd.read_csv(data_dir / 'order_products__train.csv')
orders = pd.read_csv(data_dir / 'orders.csv')
products = pd.read_csv(data_dir / 'products.csv')
aisles = pd.read_csv(data_dir / 'aisles.csv')
departments = pd.read_csv(data_dir / 'departments.csv')

tables = {
    'orders': orders,
    'order_products_prior': order_products_prior,
    'order_products_train': order_products_train,
    'products': products,
    'aisles': aisles,
    'departments': departments,
}

print(f'Loaded {len(tables)} tables from: {data_dir.resolve()}')

In [ ]:
# Create order_date by anchoring each user's latest order to a snapshot date
# Reference: The Instacart dataset snapshot is typically set to 2015-07-01
snapshot_date = pd.Timestamp('2015-07-01')

# Sort orders chronologically within each user
orders_sorted = orders.sort_values(['user_id', 'order_number']).reset_index(drop=True)

# First order (order_number=1) has no prior gap
orders_sorted['days_since_prior_order'] = orders_sorted['days_since_prior_order'].fillna(0)

# Elapsed days from each customer's first order
orders_sorted['cumulative_days'] = orders_sorted.groupby('user_id')['days_since_prior_order'].cumsum()

# Back-calculate dates so the latest order aligns with the snapshot date
max_cumulative_days = orders_sorted.groupby('user_id')['cumulative_days'].transform('max')
orders_sorted['order_date'] = snapshot_date - pd.to_timedelta(
    max_cumulative_days - orders_sorted['cumulative_days'], unit='D'
)

# Restore original row order
orders_with_dates = orders.copy()
orders_with_dates = orders_with_dates.merge(
    orders_sorted[['order_id', 'cumulative_days', 'order_date']],
    on='order_id',
    how='left'
)

print(f"✓ Created cumulative_days and order_date columns")
print(f"✓ Snapshot date: {snapshot_date}")
print(f"✓ Date range: {orders_with_dates['order_date'].min()} to {orders_with_dates['order_date'].max()}")
print(f"\nSample rows with dates:")
display(orders_with_dates[['order_id', 'user_id', 'order_number', 'days_since_prior_order', 'cumulative_days', 'order_date']].head(10))

NameError: name 'pd' is not defined

In [ ]:
# Save orders_with_dates to canonical processed/features directory
candidate_roots = [Path('.'), Path('../..')]
project_root = next(
    (p.resolve() for p in candidate_roots if (p / 'src').exists() and (p / 'data').exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not resolve project root from current notebook working directory.')

output_dir = project_root / 'data' / 'processed' / 'features'
output_dir.mkdir(parents=True, exist_ok=True)

orders_output_path = output_dir / 'orders_with_dates.csv'
orders_with_dates.to_csv(orders_output_path, index=False)

print(f"Saved orders_with_dates to: {orders_output_path}")
print(f"Rows: {len(orders_with_dates):,} | Columns: {len(orders_with_dates.columns)}")

In [ ]:
# Optional sample export for git-tracked artifact
orders_sample_path = output_dir / 'orders_with_dates_sample.csv'
orders_with_dates.sample(n=min(1000, len(orders_with_dates)), random_state=42).to_csv(
    orders_sample_path,
    index=False,
)

print(f"Saved sample artifact to: {orders_sample_path}")

## 2) Table Size and Structure Summary

In [ ]:
overview = pd.DataFrame({
    'table': list(tables.keys()),
    'rows': [df.shape[0] for df in tables.values()],
    'cols': [df.shape[1] for df in tables.values()],
    'memory_mb': [round(df.memory_usage(deep=True).sum() / 1024**2, 2) for df in tables.values()],
    'duplicate_rows': [int(df.duplicated().sum()) for df in tables.values()],
    'total_missing_cells': [int(df.isna().sum().sum()) for df in tables.values()],
}).sort_values('rows', ascending=False)

overview

## 3) Schema Snapshot
Inspect head rows and dtypes for each table.

In [ ]:
for name, df in tables.items():
    print(f'\n=== {name} ===')
    display(df.head(3))
    display(df.dtypes.rename('dtype').to_frame())

In [ ]:
table_missing = pd.DataFrame({
    'table': list(tables.keys()),
    'missing_cells': [int(df.isna().sum().sum()) for df in tables.values()],
    'missing_pct_all_cells': [round((df.isna().sum().sum() / df.size) * 100, 4) for df in tables.values()]
}).sort_values('missing_cells', ascending=False)

table_missing

### Column-level Missing Details
Show columns with missing values ranked by percentage and count.

In [ ]:
missing_by_column = (
    pd.concat(
        [
            pd.DataFrame({
                'table': name,
                'column': df.columns,
                'missing_count': df.isna().sum().values,
                'missing_pct': (df.isna().mean() * 100).values,
            })
            for name, df in tables.items()
        ],
        ignore_index=True,
    )
    .query('missing_count > 0')
    .sort_values(['missing_pct', 'missing_count'], ascending=False)
)

missing_by_column.head(20)

## 5) Basic Plots
Quick visual checks on order behavior and reorder tendency.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

orders['eval_set'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C78A8', '#F58518', '#54A24B'])
axes[0].set_title('Orders by Eval Set')
axes[0].set_xlabel('eval_set')
axes[0].set_ylabel('count')

orders['order_dow'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#72B7B2')
axes[1].set_title('Orders by Day of Week')
axes[1].set_xlabel('order_dow')
axes[1].set_ylabel('count')

orders['order_hour_of_day'].value_counts().sort_index().plot(kind='line', ax=axes[2], color='#E45756', marker='o')
axes[2].set_title('Orders by Hour of Day')
axes[2].set_xlabel('order_hour_of_day')
axes[2].set_ylabel('count')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

order_products_prior['reordered'].value_counts(normalize=True).sort_index().plot(
    kind='bar', ax=axes[0], color=['#F2CF5B', '#2E8B57']
)
axes[0].set_title('Reordered Ratio (Prior)')
axes[0].set_xlabel('reordered')
axes[0].set_ylabel('proportion')

# Top departments by number of product records
product_dept = products.merge(departments, on='department_id', how='left')
product_dept['department'].value_counts().head(10).sort_values().plot(kind='barh', ax=axes[1], color='#4C78A8')
axes[1].set_title('Top 10 Departments by Product Count')
axes[1].set_xlabel('count')
axes[1].set_ylabel('department')

plt.tight_layout()
plt.show()

## 6) Key Integrity Checks
Validate key uniqueness and relationship consistency across tables.

In [ ]:
checks = {
    'orders.order_id_unique': orders['order_id'].is_unique,
    'products.product_id_unique': products['product_id'].is_unique,
    'aisles.aisle_id_unique': aisles['aisle_id'].is_unique,
    'departments.department_id_unique': departments['department_id'].is_unique,
    'prior.order_id_in_orders_pct': round(order_products_prior['order_id'].isin(orders['order_id']).mean() * 100, 2),
    'train.order_id_in_orders_pct': round(order_products_train['order_id'].isin(orders['order_id']).mean() * 100, 2),
    'prior.product_id_in_products_pct': round(order_products_prior['product_id'].isin(products['product_id']).mean() * 100, 2),
    'train.product_id_in_products_pct': round(order_products_train['product_id'].isin(products['product_id']).mean() * 100, 2),
}

pd.Series(checks, name='result')

## 7) Findings and Next Actions
After running all cells, summarize in 4-6 bullets:
- biggest table and processing implication
- most significant missing-value pattern
- integrity check highlights
- one behavioral insight from time/reordered plots
- risks before feature engineering